In [26]:
import warnings

warnings.filterwarnings("ignore")

In [27]:
!pip install -q protobuf==3.20.3
!pip install -q transformers torch tqdm safetensors
!pip install nlpaug

In [28]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import nlpaug.augmenter.char as nac
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

In [29]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
aug = nac.KeyboardAug(aug_char_p=0.1, aug_word_p=0.2)

class TicketDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        target = self.targets[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(target, dtype=torch.float)
        }

def train_epoch(model, data_loader, optimizer, scheduler):
    model.train()
    losses = []
    
    for d in tqdm(data_loader, desc="Train Loop"):
        input_ids = d["input_ids"].to(DEVICE)
        attention_mask = d["attention_mask"].to(DEVICE)
        targets = d["targets"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=targets
        )

        loss = outputs.loss
        losses.append(loss.item())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
    return np.mean(losses)

def eval_model(model, data_loader, desc="Eval"):
    model.eval()
    final_targets = []
    final_outputs = []
    
    with torch.no_grad():
        for d in tqdm(data_loader, desc=desc):
            input_ids = d["input_ids"].to(DEVICE)
            attention_mask = d["attention_mask"].to(DEVICE)
            targets = d["targets"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
            preds = torch.sigmoid(outputs.logits).cpu().detach().numpy()
            preds = (preds > 0.5).astype(int)
            
            final_outputs.extend(preds)
            final_targets.extend(targets.cpu().detach().numpy())
            
    return np.array(final_outputs), np.array(final_targets)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FILE_PATH = '/kaggle/input/customer-support-dataset/customer_support_dataset.csv'

df = pd.read_csv(FILE_PATH)
df['instruction'] = df['instruction'].fillna("")
df['intent_list'] = df['intent'].apply(lambda x: x.split(', ') if isinstance(x, str) else [str(x)])

mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(df['intent_list'])

X = df['instruction'].values

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y_bin, 
    test_size=0.15, 
    random_state=42, 
    shuffle=True
)

relative_val_size = 0.15 / (1 - 0.15) 
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, 
    y_train_val, 
    test_size=relative_val_size, 
    random_state=42, 
    shuffle=True
)

print(f"Размеры до аугментации:\nTrain: {X_train.shape[0]}\nVal: {X_val.shape[0]}\nTest: {X_test.shape[0]}\n")

augmented_X_train = []
for text in tqdm(X_train, desc="Augmentation"):
    augmented_text = aug.augment(text)[0] 
    augmented_X_train.append(augmented_text)

augmented_X_train = np.array(augmented_X_train)

X_train = np.concatenate((X_train, augmented_X_train))
y_train = np.concatenate((y_train, y_train))

print(f"Размер Train после аугментации: {X_train.shape[0]}")

MAX_LEN = 64
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5
MODEL_NAME = 'distilbert-base-uncased'

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

train_dataset = TicketDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset = TicketDataset(X_val, y_val, tokenizer, MAX_LEN)
test_dataset = TicketDataset(X_test, y_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

labels = mlb.classes_
id2label = {idx: label for idx, label in enumerate(labels)}
label2id = {label: idx for idx, label in enumerate(labels)}

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=y_train.shape[1],
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)
model = model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

for epoch in range(EPOCHS):
    print(f'\nЭпоха {epoch + 1}/{EPOCHS}')
    train_loss = train_epoch(model, train_loader, optimizer, scheduler)
    print(f'Train Loss: {train_loss:.4f}')
    
    y_pred_val, y_true_val = eval_model(model, val_loader, desc="Validation")
    val_f1 = f1_score(y_true_val, y_pred_val, average='weighted', zero_division=0)
    print(f'Validation Weighted F1: {val_f1:.4f}')

y_pred_test, y_true_test = eval_model(model, test_loader, desc="Testing")

print("\nРЕЗУЛЬТАТЫ ДЛЯ BERT НА ТЕСТОВОЙ ВЫБОРКЕ")
print(classification_report(y_true_test, y_pred_test, target_names=mlb.classes_, zero_division=0))

Размеры до аугментации:
Train: 28256
Val: 6055
Test: 6055



Augmenting:   0%|          | 0/28256 [00:00<?, ?it/s]

Размер Train после аугментации: 56512


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Эпоха 1/3


...

Train Loss: 0.0050


Validation:   0%|          | 0/190 [00:00<?, ?it/s]

Validation Weighted F1: 0.9817


Testing:   0%|          | 0/190 [00:00<?, ?it/s]


РЕЗУЛЬТАТЫ ДЛЯ BERT НА ТЕСТОВОЙ ВЫБОРКЕ
                          precision    recall  f1-score   support

            cancel_order       0.99      1.00      0.99       246
            change_order       0.98      0.95      0.97       228
 change_shipping_address       0.97      0.95      0.96       210
  check_cancellation_fee       1.00      1.00      1.00       258
           check_invoice       0.93      0.93      0.93       217
   check_payment_methods       1.00      0.99      0.99       248
     check_refund_policy       0.99      0.97      0.98       262
               complaint       0.99      0.96      0.98       193
contact_customer_service       1.00      0.96      0.98       240
     contact_human_agent       0.96      0.98      0.97       249
          create_account       0.99      0.97      0.98       202
          delete_account       1.00      0.99      0.99       222
        delivery_options       1.00      0.99      0.99       238
         delivery_period       0.9

In [34]:
SAVE_DIR = "distilbert_ticket_intent_model"

model.save_pretrained(SAVE_DIR, safe_serialization=True)

tokenizer.save_pretrained(SAVE_DIR)
print("Веса лежат в файле model.safetensors, а классы зашиты в config.json.")

Веса лежат в файле model.safetensors, а классы зашиты в config.json.
